# Filter Bahasa pada Fitur Semantik

Notebook ini menerapkan filter bahasa pada kolom `overview` sebelum encoding SBERT. Tujuannya menghasilkan dataset eksperimen yang seluruh overview-nya berbahasa Inggris.

Isi notebook:

1. deteksi bahasa untuk seluruh overview,
2. penentuan status bahasa per film,
3. pembuangan film non-English,
4. pembuangan film tanpa overview,
5. pembentukan ulang `semantic_text`,
6. penyimpanan dataset final dalam format Parquet dan CSV.

Evaluasi similarity, fusion alpha, dan recommendation engine belum termasuk di sini.


## Tujuan dan Ruang Lingkup

Model SBERT yang dipakai berfokus pada bahasa Inggris. Karena itu, overview non-English perlu dipisahkan agar tidak mengganggu representasi semantik.

Dua keputusan metodologis yang dipakai:

- Hanya `overview` yang difilter. `tagline` dan `keywords` belum difilter pada tahap ini.
- Film yang overview-nya kosong juga dibuang, sesuai aturan bahwa dataset final tidak boleh memiliki overview kosong.

Baseline 396.254 film tetap disimpan di `movies_experiment.parquet`. Dataset baru disimpan dengan nama berbeda agar baseline tidak tertimpa.


## 1. Load Dataset dan Konfigurasi Detektor

Dataset yang dipakai adalah `movies_experiment.parquet` berisi 396.254 film yang memiliki
`semantic_text`. File `movie_ids.npy` dipakai untuk memastikan urutan baris masih sejajar
dengan embedding baseline.


In [1]:
import time

import numpy as np
import pandas as pd
from langdetect import DetectorFactory, LangDetectException, detect_langs

from movie_recommender.utils.paths import (
    DATA_PROCESSED_DIR,
    SEMANTIC_BASELINE_ARTIFACTS_DIR,
)

DetectorFactory.seed = 0  # hasil deterministik

movies = pd.read_parquet(
    DATA_PROCESSED_DIR / "movies_experiment.parquet"
)
movie_ids = np.load(SEMANTIC_BASELINE_ARTIFACTS_DIR / "movie_ids.npy")

print("rows:", len(movies))
print("embedding ids:", len(movie_ids))
print("aligned:", np.array_equal(movies["id"].to_numpy(), movie_ids))

rows: 396254
embedding ids: 396254
aligned: True


Output di atas menunjukkan dataset eksperimen dan embedding baseline memiliki jumlah
baris yang sama, yaitu 396.254, dan urutannya identik. Detektor bahasa disetel dengan
`DetectorFactory.seed = 0` agar hasilnya deterministik ketika notebook dijalankan ulang.


## 2. Fungsi Deteksi Bahasa dan Benchmark

Fungsi `detect_language` mengembalikan kode bahasa dan nilai keyakinan dari `langdetect`.
Teks kosong dikembalikan sebagai `None` karena tidak ada yang bisa dideteksi.

Sebelum memproses seluruh dataset, deteksi diuji pada sampel 1.000 film untuk mengukur
kecepatan. Hasil benchmark dipakai untuk memperkirakan durasi proses penuh, bukan untuk
menarik kesimpulan tentang akurasi.


In [2]:
def detect_language(text: str) -> tuple[str | None, float | None]:
    if not isinstance(text, str) or not text.strip():
        return None, None

    try:
        candidates = detect_langs(text)
    except LangDetectException:
        return None, None

    top = candidates[0]
    return top.lang, float(top.prob)


benchmark = movies.sample(1000, random_state=42)

start = time.perf_counter()
benchmark_results = [
    detect_language(text)
    for text in benchmark["overview"].fillna("").astype(str)
]
elapsed = time.perf_counter() - start

print("elapsed:", round(elapsed, 2), "s")
print(
    "estimated full:",
    round(elapsed / len(benchmark) * len(movies) / 60, 1),
    "minutes",
)
print(pd.Series([language for language, _ in benchmark_results]).value_counts().head(15))

elapsed: 2.12 s
estimated full: 14.0 minutes
en    975
ro      2
de      1
fr      1
af      1
et      1
Name: count, dtype: int64


Benchmark menunjukkan estimasi sekitar 14 menit untuk seluruh dataset. Angka ini
terbukti akurat: proses penuh pada bagian berikutnya memakan waktu sekitar 791 detik.

Perhatikan bahwa pada sampel ini 975 dari 1.000 overview terdeteksi sebagai bahasa Inggris.
Angka tersebut hanya gambaran awal, karena sampel belum dipisahkan berdasarkan panjang teks.


## 3. Deteksi Bahasa Seluruh Overview

Deteksi dijalankan untuk setiap film. Hasilnya berupa tabel dengan kolom `id`,
`overview_language`, dan `language_confidence`.

Film dengan overview kosong menghasilkan `None` pada kedua kolom. Film tersebut tidak
dibuang pada tahap ini, karena keputusan pembuangan dilakukan setelah status bahasa
ditentukan.


In [3]:
records = []
start = time.perf_counter()

for number, (movie_id, overview) in enumerate(
    zip(movies["id"], movies["overview"].fillna("").astype(str)),
    start=1,
):
    language, confidence = detect_language(overview)
    records.append((int(movie_id), language, confidence))

    if number % 50000 == 0:
        print(f"processed {number}/{len(movies)}")

elapsed = time.perf_counter() - start

language_df = pd.DataFrame(
    records,
    columns=["id", "overview_language", "language_confidence"],
)

print("elapsed:", round(elapsed, 2), "s")
print(language_df.head())

processed 50000/396254
processed 100000/396254
processed 150000/396254
processed 200000/396254
processed 250000/396254
processed 300000/396254
processed 350000/396254
elapsed: 790.98 s
       id overview_language  language_confidence
0   27205                en             0.999995
1  157336                en             0.999997
2     155                en             0.999996
3   19995                en             0.999996
4   24428                en             0.999996


Distribusi awal menunjukkan 385.615 film terdeteksi sebagai bahasa Inggris dan 7.479 film
tidak memiliki overview. Sisanya tersebar pada puluhan bahasa lain.

Angka ini jauh lebih kecil dibanding perkiraan berbasis `original_language`, yang sebelumnya
menunjukkan sekitar 42 persen film berlabel non-Inggris. Perbedaan tersebut terjadi karena
`original_language` adalah bahasa asli film, bukan bahasa yang dipakai pada overview.


## 4. Menentukan Status Bahasa

Dataset dan hasil deteksi dimuat ulang dari file supaya proses deteksi tidak perlu diulang
ketika notebook dibuka kembali. Bagian ini juga menghitung panjang setiap overview, karena
panjang teks dipakai sebagai syarat filter.


In [30]:
import numpy as np
import pandas as pd

from movie_recommender.utils.paths import (
    DATA_PROCESSED_DIR,
    SEMANTIC_BASELINE_ARTIFACTS_DIR,
)

language_dir = DATA_PROCESSED_DIR / "language_audit"

movies = pd.read_parquet(
    DATA_PROCESSED_DIR / "movies_experiment.parquet"
)
language_df = pd.read_parquet(
    language_dir / "overview_language.parquet"
)
movie_ids = np.load(SEMANTIC_BASELINE_ARTIFACTS_DIR / "movie_ids.npy")

print("movies:", len(movies))
print("language rows:", len(language_df))
print("embedding ids:", len(movie_ids))

movies: 396254
language rows: 396254
embedding ids: 396254


Panjang overview dihitung dari dataset asli, bukan dari hasil deteksi, sehingga nilainya
tidak bergantung pada output detektor.


### 4.1 Aturan Status Bahasa

Status ditentukan dengan tiga syarat berikut.

| Status | Syarat |
|---|---|
| `unknown` | Overview kosong atau gagal dideteksi |
| `english` | Terdeteksi `en` |
| `non_english` | Bukan `en`, keyakinan minimal 0.90, dan panjang overview minimal 150 karakter |
| `uncertain` | Bukan `en`, tetapi keyakinan atau panjang teks belum memenuhi syarat |

Syarat panjang dan keyakinan bukan angka yang dipilih sembarangan. Validasi sampel
menunjukkan akurasi detektor berbeda jauh menurut panjang teks:

| Panjang overview | Kesesuaian detektor |
|---|---:|
| Kurang dari 40 karakter | 34 persen |
| 40 sampai 80 karakter | 62,5 persen |
| 80 sampai 150 karakter | 84 persen |
| 150 karakter atau lebih | 92,5 persen |

Pada teks pendek, detektor sering salah. Contohnya overview seperti "Film by Seitz." atau
"Plot Unavailable." tetap berbahasa Inggris, tetapi ditebak sebagai bahasa Jerman atau
Prancis. Karena itu, teks pendek tidak dipakai untuk memutuskan pembuangan film.

Status `uncertain` sengaja dipertahankan. Membuangnya berisiko menghapus film berbahasa
Inggris yang overview-nya hanya pendek.


In [31]:
MIN_CHARS = 150
MIN_CONFIDENCE = 0.90

overview_chars = (
    movies.set_index("id")["overview"]
    .fillna("").astype(str).str.strip().str.len()
)

language_df = language_df.copy()
language_df["overview_chars"] = language_df["id"].map(overview_chars)

language = language_df["overview_language"]
confidence = language_df["language_confidence"]
chars = language_df["overview_chars"]

language_df["language_status"] = np.select(
    [
        language.isna(),
        language.eq("en"),
        language.ne("en") & confidence.ge(MIN_CONFIDENCE) & chars.ge(MIN_CHARS),
    ],
    ["unknown", "english", "non_english"],
    default="uncertain",
)

print(language_df["language_status"].value_counts())

language_status
english        385615
unknown          7479
uncertain        2618
non_english       542
Name: count, dtype: int64


## 5. Membuang Film Non-English

Film dengan status `non_english` dibuang seluruh barisnya, bukan hanya dikosongkan
overview-nya. Kolom `language_status` ditempelkan ke dataset agar hasil filter dapat diaudit
di kemudian hari.


In [32]:
status_by_id = language_df.set_index("id")["language_status"]
movies = movies.copy()
movies["language_status"] = movies["id"].map(status_by_id)

non_english_ids = set(
    language_df.loc[
        language_df["language_status"].eq("non_english"), "id"
    ]
)

kept = movies.loc[~movies["id"].isin(non_english_ids)].reset_index(drop=True)

print("non-English dibuang:", len(non_english_ids))
print("sebelum:", len(movies))
print("sesudah:", len(kept))

non-English dibuang: 542
sebelum: 396254
sesudah: 395712


Sebanyak 542 film dibuang pada tahap ini. Dataset turun dari 396.254 menjadi 395.712 baris.

Jumlah ini kecil, sekitar 0,137 persen, dan konsisten dengan temuan bahwa sebagian besar
overview memang sudah berbahasa Inggris.


## 6. Membuang Film Tanpa Overview

Aturan penelitian menetapkan bahwa dataset final tidak boleh memiliki overview kosong.
Karena itu, film yang overview-nya kosong ikut dibuang meskipun film tersebut mungkin masih
memiliki tagline atau keywords.

Sebelum dibuang, distribusi statusnya ditampilkan untuk memastikan yang terbuang memang
berstatus `unknown`, bukan film yang salah diklasifikasikan.


In [33]:
empty_overview = kept["overview"].fillna("").astype(str).str.strip().eq("")

print("overview kosong:", int(empty_overview.sum()))
print(kept.loc[empty_overview, "language_status"].value_counts())

kept = kept.loc[~empty_overview].reset_index(drop=True)

print("sesudah buang overview kosong:", len(kept))

overview kosong: 7471
language_status
unknown    7471
Name: count, dtype: int64
sesudah buang overview kosong: 388241


Sebanyak 7.471 film dibuang, seluruhnya berstatus `unknown`. Dataset menjadi 388.241 baris.

Delapan film berstatus `unknown` lainnya tetap tersimpan karena overview-nya tidak kosong,
hanya gagal dideteksi. Kemungkinan isinya berupa angka, simbol, atau teks yang terlalu pendek.


## 7. Membentuk Ulang semantic_text

Karena `overview` berubah, kolom `semantic_text` harus dibangun ulang dari data yang tersisa.
Fungsi `build_semantic_text` menggabungkan `overview`, `tagline`, dan `keywords` yang masih
tersedia.

Film yang setelah pembuangan tidak memiliki teks sama sekali akan dibuang pada tahap ini,
karena tidak ada konten yang bisa di-encode.


In [34]:
from movie_recommender.preprocessing.text import build_semantic_text

kept["semantic_text"] = build_semantic_text(kept)

empty_semantic = kept["semantic_text"].str.strip().eq("")
print("semantic_text kosong:", int(empty_semantic.sum()))

kept = kept.loc[~empty_semantic].reset_index(drop=True)
print("final rows:", len(kept))

semantic_text kosong: 0
final rows: 388241


Hasilnya tidak ada `semantic_text` kosong. Ini konsekuensi dari aturan sebelumnya: setiap
film yang tersisa dijamin memiliki overview, sehingga selalu ada minimal satu bagian teks.


## 8. Verifikasi Aturan Sebelum Menyimpan

Bagian ini memeriksa tiga aturan tanpa menulis file apa pun:

1. seluruh overview terisi,
2. tidak ada baris berstatus `non_english`,
3. seluruh movie ID unik.

Pemeriksaan dilakukan sebelum penyimpanan agar file yang ditulis tidak perlu diperbaiki
setelahnya.


In [35]:
overview_ok = kept["overview"].fillna("").astype(str).str.strip().ne("").all()
no_non_english = not kept["language_status"].eq("non_english").any()
ids_unique = kept["id"].is_unique

print("semua overview terisi:", bool(overview_ok))
print("tidak ada non_english:", bool(no_non_english))
print("ID unik:", bool(ids_unique))
print("rows:", len(kept))

semua overview terisi: True
tidak ada non_english: True
ID unik: True
rows: 388241


## 9. Menyimpan Dataset English-Only

Dataset disimpan sebagai `movies_experiment_en.parquet`. Nama file berbeda dari
`movies_experiment.parquet` agar baseline lama tetap utuh.

Dataset ini yang nantinya dipakai sebagai populasi eksperimen, termasuk ketika fitur
metadata dibangun pada Phase 4.


In [36]:
output_path = DATA_PROCESSED_DIR / "movies_experiment_en.parquet"

kept.to_parquet(output_path, index=False, compression="zstd")

print("saved:", output_path)
print("rows:", len(kept), "| cols:", len(kept.columns))

saved: D:\College Life\Skripsiw\hybrid-movie-recommender\data\processed\movies_experiment_en.parquet
rows: 388241 | cols: 31


Dataset tersimpan dengan 388.241 baris dan 31 kolom. Satu kolom tambahan dibanding dataset
awal adalah `language_status`, yang disimpan untuk keperluan audit.

Kolom tersebut tidak boleh dipakai sebagai fitur metadata pada Phase 4.


## 10. Ekspor Parquet dan CSV

Bagian ini berisi kode script `scripts/04_export_final_dataset.py`, bukan logika notebook.
Script tersebut mengekspor dataset yang sama ke dua format sekaligus.

Sebaiknya script dijalankan dari terminal, bukan dari dalam notebook:

```powershell
uv run python scripts/04_export_final_dataset.py
```

Script memeriksa empat aturan sebelum menulis: ID unik, overview tidak kosong,
`semantic_text` tidak kosong, dan tidak ada baris `non_english`. Jika salah satu gagal,
script berhenti tanpa menulis file.


In [37]:
"""Export the final English-only experiment dataset."""

import pandas as pd

from movie_recommender.utils.paths import DATA_PROCESSED_DIR

SOURCE = DATA_PROCESSED_DIR / "movies_experiment_en.parquet"
FINAL_PARQUET = DATA_PROCESSED_DIR / "movies_final.parquet"
FINAL_CSV = DATA_PROCESSED_DIR / "movies_final.csv"


def main() -> None:
    movies = pd.read_parquet(SOURCE)

    if not movies["id"].is_unique:
        raise ValueError("Duplicate movie IDs in final dataset")

    overview = movies["overview"].fillna("").astype(str).str.strip()
    if overview.eq("").any():
        raise ValueError("Final dataset contains empty overview")

    semantic = movies["semantic_text"].fillna("").astype(str).str.strip()
    if semantic.eq("").any():
        raise ValueError("Final dataset contains empty semantic_text")

    if movies["language_status"].eq("non_english").any():
        raise ValueError("Final dataset contains non_english rows")

    movies.to_parquet(FINAL_PARQUET, index=False, compression="zstd")
    movies.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")

    reloaded = pd.read_parquet(FINAL_PARQUET)
    pd.testing.assert_frame_equal(reloaded, movies)

    csv_ids = pd.read_csv(FINAL_CSV, usecols=["id"])

    print("rows:", len(movies))
    print("columns:", len(movies.columns))
    print("csv rows:", len(csv_ids))
    print("saved:", FINAL_PARQUET)
    print("saved:", FINAL_CSV)


if __name__ == "__main__":
    main()

rows: 388241
columns: 31
csv rows: 388241
saved: D:\College Life\Skripsiw\hybrid-movie-recommender\data\processed\movies_final.parquet
saved: D:\College Life\Skripsiw\hybrid-movie-recommender\data\processed\movies_final.csv


Kedua file berisi 388.241 baris. Parquet dipakai oleh pipeline, sedangkan CSV dipakai untuk
pemeriksaan manual atau lampiran.

CSV ditulis dengan `encoding="utf-8-sig"` agar karakter non-ASCII tetap terbaca benar ketika
dibuka di Excel. Parquet dibaca ulang dan dibandingkan dengan objek aslinya menggunakan
`assert_frame_equal` untuk memastikan tidak ada kolom atau tipe data yang berubah.


## 11. Verifikasi Hasil Simpan

File Parquet dibaca ulang dan diperiksa dengan empat assertion. Tujuannya memastikan file
yang tersimpan di disk benar-benar memenuhi aturan, bukan hanya objek di memori.


In [38]:
check = pd.read_parquet(output_path)

assert check["id"].is_unique
assert check["overview"].fillna("").astype(str).str.strip().ne("").all()
assert not check["language_status"].eq("non_english").any()
assert check["semantic_text"].str.strip().ne("").all()

print("verifikasi OK:", check.shape)

verifikasi OK: (388241, 31)


Keempat pemeriksaan lolos dengan bentuk 388.241 baris dan 31 kolom. Ini berarti dataset final
sudah konsisten dengan aturan filter.


## 12. Perbandingan dengan Baseline

Bagian ini membandingkan ID dataset final dengan ID embedding baseline. Tujuannya mengetahui
berapa film yang hilang akibat filter dan memastikan tidak ada ID baru yang tidak dikenal
baseline.


In [39]:
old_ids = set(movie_ids)
new_ids = set(check["id"])

print("baseline:", len(old_ids))
print("dataset baru:", len(new_ids))
print("tidak ada di dataset baru:", len(old_ids - new_ids))
print("ID baru tanpa embedding lama:", len(new_ids - old_ids))

baseline: 396254
dataset baru: 388241
tidak ada di dataset baru: 8013
ID baru tanpa embedding lama: 0


## 13. Ringkasan dan Catatan Metodologis

| Tahap | Jumlah baris |
|---|---:|
| Baseline eksperimen | 396.254 |
| Setelah membuang film non-English | 395.712 |
| Setelah membuang overview kosong | 388.241 |
| Setelah membentuk ulang semantic_text | 388.241 |

Komposisi status bahasa pada dataset final:

| Status | Jumlah |
|---|---:|
| `english` | 385.615 |
| `uncertain` | 2.618 |
| `unknown` | 8 |


Catatan yang perlu dibawa ke dokumentasi penelitian:

1. Filter hanya menyentuh `overview`. `tagline` dan `keywords` belum difilter, sehingga
   `semantic_text` pada 178 film masih dapat mengandung teks non-Inggris dari dua kolom
   tersebut.

2. Status `uncertain` dipertahankan karena detektor tidak dapat dipercaya pada overview
   pendek. Jumlahnya 2.618 film dan perlu dilaporkan sebagai bagian dari keterbatasan.

3. Delapan film berstatus `unknown` tetap tersimpan karena overview-nya tidak kosong,
   hanya gagal dideteksi.

4. Dataset final belum memiliki embedding. Embedding yang tersedia masih berukuran
   396.254 baris untuk baseline di `artifacts/semantic_baseline/`, sehingga perlu
   encoding ulang ke `artifacts/semantic/` tanpa menimpa baseline.

5. Encoding ulang tersebut tetap merupakan inference dengan model pretrained yang sama,
   bukan training atau fine-tuning.
